In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Copy-on-Write vs Merge-on-Read — Introduction

## The core problem

Parquet files are **immutable** — they can't be edited in place. So traditionally, updating or deleting even a single row out of a million-row file means: read the whole file → apply the change → rewrite the entire file back. For small edits on large files, this is extremely wasteful.

Delta Lake offers two strategies for handling this, controlled by whether **deletion vectors** are enabled on the table.

<img src="https://github.com/afaqueahmad7117/databricks-masterclass/blob/main/delta_lake/docs/images/Copy%20On%20Write%20&%20Merge%20On%20Read.png?raw=true" height=600/>

## Copy-on-Write (COW)

- Followed when **deletion vectors are disabled** (Delta's traditional/default behavior).
- Every update, delete, or merge **rewrites an entirely new Parquet file** containing the post-change data. The old file is marked removed in the transaction log.
- Simple and predictable, but costly — a one-row change to a 10 million-row file still means rewriting all 10 million rows into a new file.

## Merge-on-Read (MOR) — via Deletion Vectors

- Followed when **deletion vectors are enabled**.
- The original Parquet file is left **completely untouched**. Instead, the rows to be deleted/changed are recorded in a small separate **deletion vector** file — essentially a bitmap saying "skip these row positions."
- When the table is read, Delta reads the original file **and** checks the deletion vector, filtering out marked rows on the fly. An update is really just a delete (via deletion vector) + insert (small new file with just the changed row).

## Why Merge-on-Read is superior (for the right workloads)

- **Avoids full-file rewrites** — for small changes, only a tiny deletion vector file is written, not a fresh copy of the whole dataset.
- **Much lower write latency** — especially valuable for tables with frequent, small updates/deletes.
- **Less wasted I/O and compute** — no re-reading and re-writing of unchanged rows.

## The trade-off — neither is a universal "better" choice

- **COW** is better for **read-heavy** tables with infrequent writes — readers always get one clean, fully-materialized file per data segment, no extra filtering overhead at read time.
- **MOR** (deletion vectors) is better for **write-heavy / frequently-updated** tables — but reads pay a small cost of applying the deletion vector filter, and over time many small deletion vectors/files can accumulate (cleaned up later via `OPTIMIZE` + `VACUUM`).


So MOR isn't strictly "superior" — it's the better default for update-heavy workloads, which is exactly why Delta enables deletion vectors by default in modern versions.


## COW vs MOR — Quick Decision Guide

| Scenario | Update pattern | Best fit |
|---|---|---|
| **Near-real-time / streaming-style updates** | Small % of rows touched per write (e.g. ~0.5%), happening frequently | **MOR** — avoids rewriting huge files for tiny, frequent changes |
| **Batch loads / full recomputes** | Large % of rows touched (10%+) or full table overwrite, infrequent | **COW** — one clean rewrite is cheap relative to how rarely it happens, and reads stay fast |

## Rule of thumb
- **Small change, high frequency** → MOR (many small sticky-notes beats many full rewrites)
- **Large change, low frequency** → COW (one clean rewrite is cheap when it's rare, and every read afterward is effortless)

## Why the % matters
- Deletion vectors help most when a write touches a **small fraction** of an already-large file — the "note it and skip rewriting" trick only pays off when the alternative (rewriting) is wildly disproportionate to the actual change.
- Once updates start touching a large chunk of the table anyway, you're close to needing a full rewrite regardless — so COW's upfront clean rewrite stops being "wasteful" and MOR's benefit shrinks.

## Don't forget
Even MOR tables need periodic `OPTIMIZE` to consolidate accumulated deletion vectors back into clean files — MOR *delays* the rewrite cost, it doesn't eliminate it forever.


## COPY on write

In [31]:
source_df_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/raw_data/invoices_1_100.parquet"

target_df_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/deletion_vector/COPY_ON_WRITE"

source_df = spark.read.format('parquet').load(source_df_path)

source_df.write.format('delta').mode('overwrite').save(target_df_path)

In [32]:
delta_table = DeltaTable.forPath(spark, target_df_path)

In [33]:
delta_table.toDF().filter(F.col("customer_id")==5).show()

Now let's update on of the records

In [34]:
delta_table.update(
    condition = "customer_id = 5",
    set = { "age": "25" }
)

In [35]:
delta_table.toDF().filter(F.col("customer_id")==5).show()

delete operation

In [36]:
delta_table.delete(
    condition = "customer_id = 99"
)

In [37]:
delta_table.toDF().filter(F.col("customer_id")==99).show()

In [38]:
display(delta_table.history())

## Merge On Read 

Default Delta Lake Property

In [50]:
target_mor_df_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/deletion_vector/MERGE_ON_WRITE"

(
    source_df.writeTo(f"delta.`{target_mor_df_path}`")
    .using("delta")
    .tableProperty("delta.enableDeletionVectors", "true")
    .createOrReplace()
)

delta_mor_table = DeltaTable.forPath(spark, target_mor_df_path)

reader version 1 / writer version 2 is the baseline Delta protocol with no support for deletion vectors at all.

Deletion vectors require reader version 3

In [51]:
display(delta_mor_table.detail().select('format','properties'))

In [52]:
## Now delete or update records

delta_mor_table.delete(
    condition='customer_id=5'
)

delta_mor_table.update(
    condition='customer_id=99',
    set = {'age': "66"}
)

In [53]:
display(delta_mor_table.history())

## Cleaning Up History in Delta Lake

Clean up the history using `OPTIMIZE` and `VACCUM`

In [54]:
delta_mor_table.optimize().executeCompaction()

**At this point OPTIMIZE has created new, larger files — but the old small files are still sitting in storage, just marked removed in the log, not physically deleted yet.**

In [55]:
display(delta_mor_table.history())

#### VACUUM (physically remove tombstoned files)

By default, VACUUM refuses to delete anything younger than 7 days (168 hours) — a safety net against accidentally breaking time travel or concurrent readers.

To force a shorter retention (e.g. clear everything immediately), you must first disable the safety check:

In [57]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

delta_mor_table.vacuum(0)   # retain 0 hours — removes ALL tombstoned files immediately

## delta_mor_table.vacuum()   # safe default — only removes files tombstoned >7 days ago

In [58]:
display(delta_mor_table.history())

Recommended real-world order: OPTIMIZE first (compact into larger files) → then VACUUM (physically remove what OPTIMIZE just made obsolete). Running VACUUM without a prior OPTIMIZE just cleans up whatever's already been tombstoned by past updates/deletes/merges — it won't compact anything on its own.